# Checkpoint 1 — Minería de Datos
## Desempeño educativo en las pruebas Saber 11

**Integrantes:** Juan David Romero, Cesar Toro y Oscar Colorado  
**Programa:** Ingeniería de Software  
**Asignatura:** Minería de Datos  
**Corte:** 2 — Checkpoint 1

### Pregunta de decisión
¿Qué características territoriales e institucionales se relacionan con mejores o peores desempeños en las pruebas Saber 11, para identificar dónde deberían priorizarse acciones de mejoramiento educativo?

### Alcance y advertencia metodológica
La fuente ICFES documentada en el notebook anterior contiene **63 instituciones de 12 municipios de Risaralda**, con categorías anuales de desempeño entre 2014 y 2024. La fuente DANE incorporada en el cruce anterior es un **catálogo de identificación municipal** (código y nombre), no una base con matrícula, docentes o TIC. Por tanto, este análisis no representa a toda Colombia y no permite probar efectos de recursos escolares.

Las categorías A+, A, B, C y D son ordinales. Para algunos cálculos se utilizará una codificación numérica explícita (D=1, C=2, B=3, A=4, A+=5) **solo como aproximación ordinal para comparar patrones**; no son puntajes oficiales ICFES ni distancias equivalentes entre categorías. `NR` y los valores faltantes no se convierten en puntajes.

> **Cómo ejecutar:** conserva esta libreta dentro de `notebooks/`. Deben existir los archivos fuente descritos en la sección 1 dentro de `data/raw/`, o el archivo integrado en `data/processed/`. Ejecuta las celdas en orden con *Run All*.

## 1. Librerías, rutas y carga de datos

El notebook intenta primero cargar el dataset integrado que generó el notebook de adquisición. Si no existe, lee los dos CSV originales y reproduce el cruce municipal. No se inventan registros ni se reconstruyen datos ausentes.

In [ ]:
from pathlib import Path
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# El notebook está pensado para ejecutarse desde la carpeta notebooks/.
CWD = Path.cwd()
CANDIDATE_ROOTS = [CWD, CWD.parent, CWD.parent.parent]
PROJECT_ROOT = next((p for p in CANDIDATE_ROOTS if (p / "data").exists()), CWD.parent)
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

ICFES_PATH = RAW_DIR / "Pruebas_ICFES_20260929.csv"
DANE_PATH = RAW_DIR / "dane_municipios_2024.csv"
MERGED_PATH = PROCESSED_DIR / "saber11_dane_merged.csv"

print("Raíz del proyecto:", PROJECT_ROOT.resolve())
print("Dataset integrado disponible:", MERGED_PATH.exists())
print("ICFES disponible:", ICFES_PATH.exists())
print("DANE disponible:", DANE_PATH.exists())

### 1.1 Cargar dataset integrado o reproducir el cruce

In [ ]:
def normalizar_texto(valor):
    """Normaliza nombres geográficos para facilitar el cruce entre fuentes."""
    if pd.isna(valor):
        return np.nan
    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor).encode("ascii", "ignore").decode("ascii")
    return re.sub(r"\s+", " ", valor)

if MERGED_PATH.exists():
    df_original = pd.read_csv(MERGED_PATH, low_memory=False, dtype={"CODIGO_DANE_MUNICIPIO": str})
    print("Se cargó el archivo integrado:", MERGED_PATH.name)
else:
    if not ICFES_PATH.exists() or not DANE_PATH.exists():
        raise FileNotFoundError(
            "Faltan archivos de entrada. Coloca en data/raw/:\n"
            "1) Pruebas_ICFES_20260929.csv\n"
            "2) dane_municipios_2024.csv\n"
            "O ejecuta primero 01_adquisicion_cruce_diccionario.ipynb para generar "
            "data/processed/saber11_dane_merged.csv."
        )
    df_icfes = pd.read_csv(ICFES_PATH, low_memory=False)
    df_dane = pd.read_csv(DANE_PATH, low_memory=False, dtype={"CODIGO_DANE_MUNICIPIO": str})

    required_icfes = {"MUNICIPIO", "INSTITUCION EDUCATIVA"}
    required_dane = {"MUNICIPIO_DANE", "CODIGO_DANE_MUNICIPIO", "DEPARTAMENTO"}
    if not required_icfes.issubset(df_icfes.columns):
        raise ValueError(f"El CSV ICFES no contiene las columnas requeridas: {required_icfes - set(df_icfes.columns)}")
    if not required_dane.issubset(df_dane.columns):
        raise ValueError(f"El CSV DANE no contiene las columnas requeridas: {required_dane - set(df_dane.columns)}")

    year_cols_raw = [c for c in df_icfes.columns if re.fullmatch(r"AÑO\s+20\d{2}", str(c).strip(), flags=re.IGNORECASE)]
    if not year_cols_raw:
        raise ValueError("No se encontraron columnas anuales con formato 'AÑO 2014', 'AÑO 2015', etc.")
    df_icfes["MUNICIPIO_KEY"] = df_icfes["MUNICIPIO"].map(normalizar_texto)
    df_dane["MUNICIPIO_KEY"] = df_dane["MUNICIPIO_DANE"].map(normalizar_texto)
    df_icfes["MUNICIPIO_KEY"] = df_icfes["MUNICIPIO_KEY"].replace({"santa rosa": "santa rosa de cabal"})
    for col in year_cols_raw:
        df_icfes[col] = df_icfes[col].astype("string").str.strip().str.upper().replace({"NR": "NR", "NAN": pd.NA, "": pd.NA})
    if df_dane["MUNICIPIO_KEY"].duplicated().any():
        raise ValueError("La llave municipal de DANE no es única; revisar antes del merge.")
    df_original = df_icfes.merge(df_dane, on="MUNICIPIO_KEY", how="left", validate="many_to_one", indicator=True)
    if not (df_original["_merge"] == "both").all():
        no_match = df_original.loc[df_original["_merge"] != "both", "MUNICIPIO"].drop_duplicates().tolist()
        raise ValueError(f"Hay municipios ICFES sin correspondencia DANE: {no_match}")
    df_original.to_csv(MERGED_PATH, index=False, encoding="utf-8-sig")
    print("Cruce municipal reproducido y guardado en:", MERGED_PATH)

# Estandarizar columnas anuales y categorías sin alterar el dato fuente.
year_cols = sorted(
    [c for c in df_original.columns if re.fullmatch(r"AÑO\s+20\d{2}", str(c).strip(), flags=re.IGNORECASE)],
    key=lambda x: int(re.search(r"20\d{2}", x).group())
)
if not year_cols:
    raise ValueError("El dataset no contiene columnas anuales 'AÑO YYYY'.")
for col in year_cols:
    df_original[col] = df_original[col].astype("string").str.strip().str.upper().replace({"": pd.NA, "NAN": pd.NA})

print("Dimensiones del dataset:", df_original.shape)
print("Años disponibles:", [re.search(r"20\d{2}", c).group() for c in year_cols])
display(df_original.head())

## 2. Parte 1 — Calidad de datos (7%)

Se inspeccionan estructura, tipos, faltantes, duplicados y dominios válidos. Las categorías de desempeño son variables categóricas ordinales, no fechas ni puntajes continuos.

In [ ]:
print("Filas y columnas:", df_original.shape)
print("\nTipos de datos:")
display(df_original.dtypes.astype(str).to_frame("tipo_dato"))
print("\nValores faltantes por columna:")
display(df_original.isna().sum().to_frame("faltantes").sort_values("faltantes", ascending=False))
print("\nDuplicados completos:", df_original.duplicated().sum())
print("\nInstituciones duplicadas (nombre + municipio):", df_original.duplicated(["MUNICIPIO", "INSTITUCION EDUCATIVA"]).sum())
print("\nDistribución de municipios:")
display(df_original["MUNICIPIO"].value_counts(dropna=False).to_frame("instituciones"))

### 2.1 Validación y homologación de categorías

Se admiten `A+`, `A`, `B`, `C`, `D` y `NR`, que son las categorías observadas en el notebook anterior. La variante `Nr` se homologa a `NR` porque representa la misma etiqueta. Los faltantes se conservan como faltantes: no se confunden con `NR`.

In [ ]:
CATEGORIAS_VALIDAS = {"A+", "A", "B", "C", "D", "NR"}

for col in year_cols:
    df_original[col] = df_original[col].replace({"NR": "NR", "NR ": "NR", "NR. ": "NR"})
    observadas = set(df_original[col].dropna().astype(str).str.upper().unique())
    invalidas = observadas - CATEGORIAS_VALIDAS
    print(f"{col}: categorías observadas={sorted(observadas)} | no reconocidas={sorted(invalidas)}")

# Tipos correctos para identificadores y nombres: texto; código DANE conserva ceros iniciales.
for col in ["MUNICIPIO", "INSTITUCION EDUCATIVA", "MUNICIPIO_KEY", "MUNICIPIO_DANE", "CODIGO_DANE_MUNICIPIO", "DEPARTAMENTO"]:
    if col in df_original.columns:
        df_original[col] = df_original[col].astype("string").str.strip()

# Año de referencia es una etiqueta categórica; no se fuerza a numérico.
print("\nTipos después de validar:")
display(df_original.dtypes.astype(str).to_frame("tipo_dato"))

### 2.2 Valores atípicos: criterio, detección y decisión

La fuente no contiene puntajes numéricos oficiales, sino categorías ordinales. Para poder revisar posibles instituciones extremas, se crea una codificación auxiliar (D=1, C=2, B=3, A=4, A+=5) en 2021–2024. `NR` queda como valor faltante en el cálculo. Se aplica IQR al promedio ordinal por institución. Los casos detectados se **marcan, no se eliminan**, porque un desempeño alto o bajo puede ser real y no demuestra un error de captura.

In [ ]:
ORDEN_DESEMPENO = {"D": 1, "C": 2, "B": 3, "A": 4, "A+": 5}
periodo_reciente = [c for c in year_cols if re.search(r"20\d{2}", c).group() in {"2021", "2022", "2023", "2024"}]
if not periodo_reciente:
    periodo_reciente = year_cols[-min(4, len(year_cols)):]

# Copia de trabajo; no modifica las categorías originales.
df = df_original.copy()
for col in periodo_reciente:
    df[f"{col}_ORD"] = df[col].map(ORDEN_DESEMPENO).astype("float")
ordinal_cols = [f"{c}_ORD" for c in periodo_reciente]
df["PROMEDIO_ORDINAL_RECIENTE"] = df[ordinal_cols].mean(axis=1)

q1 = df["PROMEDIO_ORDINAL_RECIENTE"].quantile(0.25)
q3 = df["PROMEDIO_ORDINAL_RECIENTE"].quantile(0.75)
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr
df["ATIPICO_IQR"] = (
    (df["PROMEDIO_ORDINAL_RECIENTE"] < limite_inferior) |
    (df["PROMEDIO_ORDINAL_RECIENTE"] > limite_superior)
)

print(f"Periodo utilizado: {periodo_reciente}")
print(f"Q1={q1:.3f}; Q3={q3:.3f}; IQR={iqr:.3f}")
print(f"Límite inferior={limite_inferior:.3f}; límite superior={limite_superior:.3f}")
print("Instituciones marcadas como atípicas:", int(df["ATIPICO_IQR"].sum()))
display(df.loc[df["ATIPICO_IQR"], ["MUNICIPIO", "INSTITUCION EDUCATIVA", "PROMEDIO_ORDINAL_RECIENTE"]])

print("\nJustificación de tratamiento:")
print("1) La categoría 'Nr' se homologa a 'NR': diferencia de mayúsculas/minúsculas, mismo código categórico.")
print("2) Los faltantes se conservan como NaN: no hay evidencia para imputarlos ni equipararlos a NR.")
print("3) Los registros extremos por IQR se marcan, no se eliminan: pueden reflejar diferencias reales entre instituciones.")
print("4) No se convierten categorías en puntajes ICFES: la codificación 1–5 es solo una escala ordinal auxiliar.")

## 3. Parte 2 — Análisis exploratorio (EDA, 7%)

Se presentan estadísticas descriptivas de las variables derivadas, frecuencias de categorías y gráficos. La interpretación se limita a las 63 instituciones y 12 municipios de la muestra documentada.

In [ ]:
# Estadísticas descriptivas de las variables ordinales auxiliares.
columnas_numericas_eda = ordinal_cols + ["PROMEDIO_ORDINAL_RECIENTE"]
display(df[columnas_numericas_eda].describe().T)

# Frecuencias de categorías por año reciente.
for col in periodo_reciente:
    print(f"Frecuencias — {col}")
    display(df[col].value_counts(dropna=False).rename_axis("categoria").to_frame("frecuencia"))

# Frecuencia de instituciones por municipio.
display(df["MUNICIPIO"].value_counts().rename_axis("municipio").to_frame("instituciones"))

### 3.1 Gráfico 1 — Distribución de categorías por año

In [ ]:
frecuencias_anuales = pd.DataFrame({c: df[c].value_counts() for c in periodo_reciente}).fillna(0)
orden = [x for x in ["A+", "A", "B", "C", "D", "NR"] if x in set(df[periodo_reciente].stack().dropna())]
frecuencias_anuales = frecuencias_anuales.reindex(orden)
ax = frecuencias_anuales.T.plot(kind="bar", stacked=True, figsize=(10, 5))
ax.set_title("Distribución de categorías de desempeño por año")
ax.set_xlabel("Año")
ax.set_ylabel("Número de instituciones")
ax.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 3.2 Gráfico 2 — Promedio ordinal reciente por municipio

In [ ]:
resumen_municipio = (df.groupby("MUNICIPIO", dropna=False)["PROMEDIO_ORDINAL_RECIENTE"]
                     .agg(promedio="mean", instituciones="count")
                     .sort_values("promedio", ascending=False))
ax = resumen_municipio["promedio"].plot(kind="bar", figsize=(11, 5))
ax.set_title("Promedio ordinal de desempeño 2021–2024 por municipio")
ax.set_xlabel("Municipio")
ax.set_ylabel("Promedio ordinal (D=1, C=2, B=3, A=4, A+=5)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()
display(resumen_municipio)

### 3.3 Gráfico 3 — Distribución de desempeño por institución

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["PROMEDIO_ORDINAL_RECIENTE"].dropna(), bins=np.arange(0.5, 6.5, 0.5), edgecolor="black")
plt.title("Distribución del promedio ordinal reciente por institución")
plt.xlabel("Promedio ordinal (D=1, C=2, B=3, A=4, A+=5)")
plt.ylabel("Número de instituciones")
plt.xticks(range(1, 6))
plt.tight_layout()
plt.show()

### 3.4 Gráfico 4 — Evolución de las categorías en la muestra

In [ ]:
# Proporción de instituciones en cada categoría para cada año.
proporciones = pd.DataFrame({c: df[c].value_counts(normalize=True) for c in periodo_reciente}).T
proporciones = proporciones.reindex(columns=orden, fill_value=0)
ax = proporciones.plot(kind="line", marker="o", figsize=(10, 5))
ax.set_title("Proporción de instituciones por categoría de desempeño")
ax.set_xlabel("Año")
ax.set_ylabel("Proporción")
ax.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

### 3.5 Hallazgos conectados con la pregunta de decisión

La siguiente celda calcula indicadores concretos y redacta hallazgos con los valores reales de la base. Estos resultados describen asociaciones dentro de la muestra; no prueban causalidad ni representan a todo el país.

In [ ]:
# Indicadores calculados directamente desde el dataset.
n_instituciones = len(df)
n_municipios = df["MUNICIPIO"].nunique()
mejor_municipio = resumen_municipio["promedio"].idxmax() if resumen_municipio["promedio"].notna().any() else "No disponible"
peor_municipio = resumen_municipio["promedio"].idxmin() if resumen_municipio["promedio"].notna().any() else "No disponible"
mejor_valor = resumen_municipio["promedio"].max()
peor_valor = resumen_municipio["promedio"].min()

if "AÑO 2024" in df.columns:
    freq_2024 = df["AÑO 2024"].value_counts(normalize=True, dropna=False)
    categoria_2024 = df["AÑO 2024"].value_counts().idxmax()
    porcentaje_categoria_2024 = 100 * freq_2024.loc[categoria_2024]
else:
    categoria_2024, porcentaje_categoria_2024 = "No disponible", np.nan

if len(ordinal_cols) >= 2:
    comparacion_inicio = df[ordinal_cols[0]].mean()
    comparacion_fin = df[ordinal_cols[-1]].mean()
else:
    comparacion_inicio = comparacion_fin = np.nan

porcentaje_nr = 100 * df[periodo_reciente].eq("NR").sum().sum() / max(1, df[periodo_reciente].notna().sum().sum())

hallazgos = [
    f"Cobertura: la base contiene {n_instituciones} instituciones de {n_municipios} municipios; por tanto, los resultados corresponden a esta muestra de Risaralda y no a Colombia completa.",
    f"Diferencia territorial descriptiva: {mejor_municipio} presenta el mayor promedio ordinal ({mejor_valor:.2f}) y {peor_municipio} el menor ({peor_valor:.2f}) entre los municipios con datos válidos.",
    f"Categoría más frecuente en 2024: {categoria_2024} ({porcentaje_categoria_2024:.1f}% de los registros no faltantes de 2024).",
    f"Cambio entre primer y último año del periodo analizado ({periodo_reciente[0]} a {periodo_reciente[-1]}): el promedio ordinal pasó de {comparacion_inicio:.2f} a {comparacion_fin:.2f}; es una comparación descriptiva, no causal.",
    f"Los registros NR representan aproximadamente {porcentaje_nr:.1f}% de las observaciones no faltantes del periodo reciente; conviene tratarlos como categoría no reportada y no como bajo desempeño."
]
for i, h in enumerate(hallazgos, 1):
    print(f"{i}. {h}")

## 4. Parte 3 — Ingeniería de características y pipeline (6%)

Se generan características interpretables a partir de las categorías anuales. Las transformaciones se encapsulan en una función determinista. Se utiliza OneHotEncoder para variables categóricas y StandardScaler para las variables numéricas derivadas.

In [ ]:
def crear_caracteristicas(datos: pd.DataFrame) -> pd.DataFrame:
    """Genera características institucionales reproducibles desde categorías anuales.

    La escala D=1, C=2, B=3, A=4, A+=5 es ordinal y exploratoria; no equivale
    a puntajes oficiales. NR y faltantes no se convierten en valores numéricos.
    """
    resultado = datos.copy()
    years = sorted(
        [c for c in resultado.columns if re.fullmatch(r"AÑO\s+20\d{2}", str(c).strip(), flags=re.IGNORECASE)],
        key=lambda x: int(re.search(r"20\d{2}", x).group())
    )
    recientes = [c for c in years if re.search(r"20\d{2}", c).group() in {"2021", "2022", "2023", "2024"}]
    if not recientes:
        recientes = years[-min(4, len(years)):]
    codificadas = pd.DataFrame({c: resultado[c].map(ORDEN_DESEMPENO) for c in recientes}, index=resultado.index).astype(float)

    resultado["PROMEDIO_ORDINAL_RECIENTE"] = codificadas.mean(axis=1)
    resultado["ANIOS_CON_RESULTADO"] = codificadas.notna().sum(axis=1)
    resultado["PORCENTAJE_NR_RECIENTE"] = resultado[recientes].eq("NR").sum(axis=1) / len(recientes) * 100

    # Pendiente lineal sobre los años disponibles por institución; exige al menos 2 observaciones.
    years_num = np.array([int(re.search(r"20\d{2}", c).group()) for c in recientes], dtype=float)
    def pendiente(fila):
        mask = fila.notna().to_numpy()
        if mask.sum() < 2:
            return np.nan
        return float(np.polyfit(years_num[mask], fila.to_numpy(dtype=float)[mask], 1)[0])
    resultado["TENDENCIA_ORDINAL_ANUAL"] = codificadas.apply(pendiente, axis=1)

    # Variables categóricas para segmentación descriptiva.
    if "DEPARTAMENTO" not in resultado.columns:
        resultado["DEPARTAMENTO"] = "No informado"
    return resultado


df_features = crear_caracteristicas(df_original)
variables_derivadas = ["PROMEDIO_ORDINAL_RECIENTE", "ANIOS_CON_RESULTADO", "PORCENTAJE_NR_RECIENTE", "TENDENCIA_ORDINAL_ANUAL"]
display(df_features[["MUNICIPIO", "INSTITUCION EDUCATIVA"] + variables_derivadas].head(10))
print("Variables derivadas creadas:", variables_derivadas)

### 4.1 Preparación de variables categóricas y numéricas

In [ ]:
categoricas = [c for c in ["DEPARTAMENTO", "MUNICIPIO", "NATURALEZA", "JORNADA", "CALENDARIO"] if c in df_features.columns]
# La fuente documentada no contiene NATURALEZA/JORNADA/CALENDARIO; se utilizan solo si aparecen en futuras versiones.
if not categoricas:
    categoricas = [c for c in ["DEPARTAMENTO", "MUNICIPIO"] if c in df_features.columns]

numericas = [c for c in ["PROMEDIO_ORDINAL_RECIENTE", "ANIOS_CON_RESULTADO", "PORCENTAJE_NR_RECIENTE", "TENDENCIA_ORDINAL_ANUAL"] if c in df_features.columns]

X = df_features[categoricas + numericas].copy()
for col in categoricas:
    X[col] = X[col].astype("string").fillna("No informado")
for col in numericas:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Imputación mediana mínima para que el pipeline admita tendencias no calculables;
# el escalado se aplica después. Se registra como parte del proceso reproducible.
from sklearn.impute import SimpleImputer

preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", Pipeline(steps=[("imputar", SimpleImputer(strategy="median")),
                                     ("escalar", StandardScaler())]), numericas),
        ("categoricas", Pipeline(steps=[("imputar", SimpleImputer(strategy="most_frequent")),
                                        ("codificar", OneHotEncoder(handle_unknown="ignore"))]), categoricas)
    ],
    remainder="drop"
)
pipeline = Pipeline(steps=[("preprocesamiento", preprocesador)])

X_transformado = pipeline.fit_transform(X)
print("Variables categóricas codificadas:", categoricas)
print("Variables numéricas escaladas:", numericas)
print("Dimensiones antes:", X.shape)
print("Dimensiones después del pipeline:", X_transformado.shape)

### 4.2 Verificación de reproducibilidad

La función de ingeniería y el pipeline deben producir los mismos resultados cuando se ejecutan dos veces con los mismos datos y configuración.

In [ ]:
# Repetir ingeniería de características.
df_features_2 = crear_caracteristicas(df_original)
pd.testing.assert_frame_equal(
    df_features[variables_derivadas].reset_index(drop=True),
    df_features_2[variables_derivadas].reset_index(drop=True),
    check_dtype=False
)

# Repetir el ajuste del pipeline con el mismo conjunto X.
pipeline_2 = Pipeline(steps=[("preprocesamiento", preprocesador)])
X_transformado_2 = pipeline_2.fit_transform(X)

if hasattr(X_transformado, "toarray"):
    iguales = np.allclose(X_transformado.toarray(), X_transformado_2.toarray(), equal_nan=True)
else:
    iguales = np.allclose(X_transformado, X_transformado_2, equal_nan=True)

assert iguales, "El pipeline no produjo resultados reproducibles."
print("Validación exitosa: la ingeniería de características y el pipeline son reproducibles.")

## 5. Segmentación exploratoria de instituciones

Como complemento descriptivo, se segmentan instituciones según su promedio ordinal reciente. Los cortes se calculan mediante terciles de la muestra y no constituyen categorías oficiales del ICFES ni un ranking institucional.

In [ ]:
df_features["SEGMENTO_ORDINAL"] = pd.qcut(
    df_features["PROMEDIO_ORDINAL_RECIENTE"].rank(method="first"),
    q=3,
    labels=["Grupo ordinal inferior", "Grupo ordinal intermedio", "Grupo ordinal superior"],
    duplicates="drop"
)
segmentacion = (df_features.groupby("SEGMENTO_ORDINAL", observed=False)
                .agg(instituciones=("INSTITUCION EDUCATIVA", "count"),
                     promedio_ordinal=("PROMEDIO_ORDINAL_RECIENTE", "mean"),
                     municipios=("MUNICIPIO", "nunique"))
                .reset_index())
display(segmentacion)

ax = segmentacion.set_index("SEGMENTO_ORDINAL")["instituciones"].plot(kind="bar", figsize=(8, 4))
ax.set_title("Número de instituciones por segmento ordinal")
ax.set_xlabel("Segmento")
ax.set_ylabel("Instituciones")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 6. Exportación de resultados

Se guardan el dataset con características derivadas, la tabla de resumen territorial y la segmentación para facilitar la trazabilidad del análisis.

In [ ]:
FEATURES_PATH = PROCESSED_DIR / "saber11_checkpoint1_features.csv"
MUNICIPIO_PATH = PROCESSED_DIR / "resumen_desempeno_municipio.csv"
SEGMENTACION_PATH = PROCESSED_DIR / "segmentacion_instituciones.csv"

# Exportación sin columnas auxiliares internas del merge cuando no son necesarias.
df_features.to_csv(FEATURES_PATH, index=False, encoding="utf-8-sig")
resumen_municipio.to_csv(MUNICIPIO_PATH, encoding="utf-8-sig")
segmentacion.to_csv(SEGMENTACION_PATH, index=False, encoding="utf-8-sig")

print("Archivos exportados:")
for p in [FEATURES_PATH, MUNICIPIO_PATH, SEGMENTACION_PATH]:
    print("-", p.relative_to(PROJECT_ROOT), "|", p.stat().st_size, "bytes")

## 7. Conclusiones y limitaciones\n\n### Conclusiones

1. El análisis permite describir diferencias entre las instituciones y municipios incluidos en la muestra de Risaralda.
2. Las categorías de desempeño se trataron como ordinales; la escala numérica auxiliar facilita comparaciones, pero no representa puntajes oficiales ni supone distancias iguales entre categorías.
3. Los registros `NR` se mantuvieron diferenciados de los datos faltantes.
4. Los posibles atípicos se marcaron para revisión y no se eliminaron automáticamente, porque pueden representar desempeños reales.
5. La segmentación es exploratoria y relativa a esta muestra; no es un ranking oficial ni demuestra causalidad.

### Limitaciones

- La muestra disponible contiene 63 instituciones de 12 municipios de Risaralda; no permite generalizar los resultados a todo Colombia.
- La fuente de DANE utilizada en el cruce es un catálogo municipal. No aporta en este archivo matrícula, docentes, zona rural/urbana ni TIC.
- Los resultados son categorías anuales y no puntajes individuales por área.
- El archivo documentado llega hasta 2024; 2025 no está disponible en esta fuente.
- Para responder mejor la pregunta de decisión se requiere incorporar una base DANE EDUC con variables institucionales adicionales y verificar compatibilidad de códigos, periodos y niveles de agregación.

### Relación con la pregunta de decisión

Los resultados pueden orientar qué municipios e instituciones de esta muestra merecen un análisis más profundo. No permiten concluir que una característica territorial cause un determinado desempeño. Las acciones de mejoramiento deben apoyarse en información complementaria y en una evaluación contextual de cada institución.

## 8. Lista de verificación del Checkpoint 1\n\n| Requisito de la rúbrica | Evidencia en este notebook |\n|---|---|\n| Tipos de datos revisados/corregidos | Secciones 2 y 2.1 |\n| Atípicos identificados y decisión justificada | Sección 2.2 |\n| `describe()` y `value_counts()` | Sección 3 |\n| Mínimo tres gráficos | Secciones 3.1 a 3.4 |\n| 4–5 hallazgos conectados con negocio | Sección 3.5 |\n| Mínimo tres variables derivadas | Sección 4 |\n| Codificación categórica | Sección 4.1 — OneHotEncoder |\n| Escalado numérico | Sección 4.1 — StandardScaler |\n| Función/pipeline reproducible | Secciones 4 y 4.2 |\n\n**Antes de entregar:** ejecutar *Run All*, comprobar que no haya errores, revisar que las rutas de los CSV sean correctas y confirmar que las salidas y gráficos se hayan generado.